In [ ]:
import pandas as pd
import numpy as np
import time

from scipy.integrate import odeint
import matplotlib.pyplot as plt
import pickle

_ = np.random.seed(0)

import pyabc
from pyabc import ABCSMC, Distribution, RV, MultivariateNormalTransition, QuantileEpsilon
import tempfile

pyabc.settings.set_figure_params('pyabc') 

In [ ]:
def ode_system(y, t, beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N):
    S, E, I, H, R, D = y
    dSdt = - (beta_I * I + beta_H * H) * S / N
    dEdt = (beta_I * I + beta_H * H) * S / N - kappa * E
    dIdt =  kappa * E - (alpha + gamma_I + delta_I) * I
    dHdt = alpha * I - (gamma_H + delta_H) * H
    dRdt = gamma_I * I + gamma_H * H
    dDdt = delta_I * I + delta_H *H
    return [dSdt, dEdt, dIdt, dHdt, dRdt, dDdt]

def simulate(parameters, ic, T):
    beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H = parameters
    S0, E0, I0, H0, R0, D0 = ic

    N = S0 + E0 + I0 + H0 + R0 + D0
    y0=ic
    
    t = np.arange(T+1)
    ret = odeint(ode_system, y0, t, args=(beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N))

    cases1 = poisson_noise(kappa * ret[1:,1])
    cases2 = poisson_noise(alpha * ret[1:,2])
    cases3 = poisson_noise(delta_I * ret[1:,2] + delta_H * ret[1:,3])

    return np.stack([cases1, cases2, cases3], axis=1)


def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return with_noise

In [ ]:
N  = 100000
E0 = 0
I0 = 10
H0 = 0
R0 = 0
D0 = 0
S0 = N - E0 - I0 - H0 - R0 - D0

duration=100

init_cond = [S0, E0, I0, H0, R0, D0]

In [ ]:
with open('../../Data/Model2/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
def model(params):
    pvec = [
        params["beta_I"],
        params["beta_H"],
        params["kappa"],
        params["alpha"],
        params["gamma_I"],
        params["delta_I"],
        params["gamma_H"],
        params["delta_H"]
    ]
    
    sim = simulate(pvec, init_cond, T=100)

    return {'cases1': sim[:, 0], 'cases2': sim[:, 1], 'cases3': sim[:, 2]} 

In [ ]:
eps = QuantileEpsilon(initial_epsilon='from_sample', alpha=0.2)
transition = MultivariateNormalTransition(scaling=0.5)

In [ ]:
p_distance = pyabc.AdaptivePNormDistance(
    p=2,
    scale_function=pyabc.distance.std,
    all_particles_for_scale=True
)

In [ ]:
low  = np.array([0.7, 0.2, 0.1, 0.1, 0.1, 0.001, 0.1, 0.001])
high = np.array([1.3, 0.8, 0.3, 0.3, 0.3, 0.05, 0.4, 0.05])

interval=high-low

In [ ]:
prior = Distribution(
    beta_I = RV("uniform", low[0], interval[0]),
    beta_H = RV("uniform", low[1], interval[1]),
    kappa  = RV("uniform", low[2], interval[2]),
    alpha  = RV("uniform", low[3], interval[3]),
    gamma_I  = RV("uniform", low[4], interval[4]),
    delta_I  = RV("uniform", low[5], interval[5]),
    gamma_H  = RV("uniform", low[6], interval[6]),
    delta_H  = RV("uniform", low[7], interval[7])
)

In [ ]:
param_names = ["beta_I", "beta_H", "kappa", "alpha", "gamma_I", "delta_I", "gamma_H", "delta_H"]

In [ ]:
results = []

for i, sim in enumerate(simulation_dataset):
    obs_data = sim['poisson']
    obs_dict=obs_dict = {
        "cases1": obs_data[:, 0],
        "cases2": obs_data[:, 1],
        "cases3": obs_data[:, 2]
    }
 
    db_path = "sqlite:///" + tempfile.mkstemp(suffix=f"abc_{i}.db")[1]
    
    sampler = pyabc.sampler.MulticoreEvalParallelSampler(n_procs=16)
    abc = ABCSMC(model, prior, p_distance, sampler=sampler, population_size=1000, transitions=transition, eps=eps)
    abc.new(db_path, obs_dict)
    
    history = abc.run(max_total_nr_simulations=100000)


    df_posterior = history.get_distribution()
    
    results.append({
        "params": sim['params'],
        "posterior": df_posterior,
        "populations": history.get_all_populations()
    })

In [ ]:
posterior_samples=[]
for i, res in enumerate(results): 
    df, weights = res['posterior']
    df = df[param_names]
    
    kde_estimator = pyabc.transition.MultivariateNormalTransition()
    kde_estimator.fit(df, weights)
    
    num_samples = 10000
    samples = kde_estimator.rvs(num_samples)
    posterior_samples.append(samples)